## Script for scraping Courtvision data From Roland Garros and Australian Open

In [1]:
import pandas as pd
import numpy as np
import json
import requests
import os
import time
from fake_useragent import UserAgent
import random
import json
import base64
from datetime import datetime
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.backends import default_backend


pd.set_option('display.max_rows', 500)

# --> Import functions from "process" script
import sys
sys.path.append('../src')



## Decode the ciphered data

In [2]:
mapping = {
    "a11": "cruciality",
    "a12": "trajectoryData",
    "a13": "serverId",
    "a14": "scorerId",
    "a15": "receiverId",
    "a16": "ballSpeed",
    "a17": "returnSpeed",
    "a18": "returnSpeedFrench",
    "a93": "rallyLength",
    "a94": "maxRally",
    "a19": "spin",
    "a20": "heightAboveNet",
    "a21": "ballSpeedFrench",
    "a22": "heightAboveNetFrench",
    "a23": "distanceOutsideCourt",
    "a24": "distanceOutsideCourtFrench",
    "a95": "pointEndType",
    "a25": "strokeType",
    "a27": "ballHitCordinate",
    "a28": "ballPeakCordinate",
    "a29": "ballNetCordinate",
    "a30": "ballBounceCordinate",
    "a31": "ballLastCordinate",
    "a32": "serverCordinate",
    "a33": "receiverCordinate",
    "a34": "serveBounceCordinate",
    "a35": "scoreBoard",
    "a36": "serveDirectionId",
    "a108": "isTieBreak",
    "a109": "setWinner",
    "a86": "id",
    "a107": "pointNumber",
    "a74": "erroneousBall",
    "a37": "aces",
    "a38": "convertedBreakPoints",
    "a39": "doubleFault",
    "a40": "firstServeIn",
    "a41": "firstServePointsWon",
    "a42": "netPoints",
    "a43": "pointsWon",
    "a44": "returnPoints",
    "a45": "secondServeIn",
    "a46": "secondServePointsWon",
    "a47": "unforcedError",
    "a48": "winner",
    "a49": "statsData",
    "a50": "pointsData",
    "a51": "percentage",
    "a52": "crucialPercentage",
    "a53": "percentagePlayer",
    "a54": "percentageOpponent",
    "a55": "percentagePlayerCrucial",
    "a56": "percentageOpponentCrucial",
    "a57": "count",
    "a58": "onCourt",
    "a59": "set0",
    "a60": "set1",
    "a61": "set2",
    "a62": "set3",
    "a63": "set4",
    "a64": "set5",
    "a65": "adCourt",
    "a66": "deuceCourt",
    "a67": "percentageT",
    "a68": "percentageM",
    "a69": "percentageW",
    "a70": "x",
    "a71": "y",
    "a72": "z",
    "a73": "position",
    "a75": "isMatchComplete",
    "a76": "eventType",
    "a77": "courtName",
    "a78": "courtId",
    "a79": "playersData",
    "a80": "setsCompleted",
    "a81": "pointId",
    "a82": "matchStatus",
    "a83": "playerTeam",
    "a84": "opponentTeam",
    "a85": "name",
    "a87": "country",
    "a88": "seed",
    "a89": "returnPlacement",
    "a90": "errorType",
    "a91": "winnerPlacement",
    "a92": "unforcedErrorPlacement",
    "a96": "serveType",
    "a97": "court",
    "a98": "setNumber",
    "a99": "set",
    "a100": "game",
    "a101": "point",
    "a102": "serve",
    "a103": "hand",
    "a104": "breakPoint",
    "a26": "runAroundForeHand",
    "a105": "breakPointConverted",
    "a106": "trappedByNet",
    "a122":"playerSet1Score",
    "a123":"playerSet2Score",
    "a124":"playerSet3Score",
    "a125":"playerSet4Score",
    "a126":"playerSet5Score",
    "a127":"playerSet1TieBreakScore",
    "a128":"playerSet2TieBreakScore",
    "a129":"playerSet3TieBreakScore",
    "a130":"playerSet4TieBreakScore",
    "a131":"playerSet5TieBreakScore",
    "a132":"opponentSet1Score",
    "a133":"opponentSet2Score",
    "a134":"opponentSet3Score",
    "a135":"opponentSet4Score",
    "a136":"opponentSet5Score",
    "a137":"opponentSet1TieBreakScore",
    "a138":"opponentSet2TieBreakScore",
    "a139":"opponentSet3TieBreakScore",
    "a140":"opponentSet4TieBreakScore",
    "a141":"opponentSet5TieBreakScore",
    "a142":"playergamescore",
    "a143":"opponentgamescore",
    "a151": "playerPositionsData",
    "a152": "belowCourt",
}

def apply_mapping(wrong_structure, mapping):
    if isinstance(wrong_structure, dict):
        corrected = {}
        for key, value in wrong_structure.items():
            new_key = mapping.get(key, key)
            corrected[new_key] = apply_mapping(value, mapping)
        return corrected
    elif isinstance(wrong_structure, list):
        return [apply_mapping(item, mapping) for item in wrong_structure]
    else:
        return wrong_structure

In [3]:
def base16_to_base36(hex_num):
    decimal_num = int(hex_num, 16)
    base36_num = ''
    # alphabet = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    alphabet = '0123456789abcdefghijklmnopqrstuvwxyz'

    while decimal_num > 0:
        decimal_num, i = divmod(decimal_num, 36)
        base36_num = alphabet[i] + base36_num

    return base36_num


def custom_base24encode(number):
    """A custom base 24 encoding. Adjust this function based on your needs."""
    assert number >= 0, 'positive integer required'
    digits = '0123456789abcdefghijklm'
    res = ''
    while number:
        number, rem = divmod(number, 24)
        res = digits[rem] + res
    return res or '0'



def format_date_py(timestamp):
    """Format the date in a specific way based on the provided JavaScript logic."""
    
    dt = datetime.utcfromtimestamp(timestamp / 1000)
    timeZoneOffset = 180 # e
    

    day = dt.day # n
    r = int(str(day).zfill(2)[::-1])

    year = dt.year # i

    reversed_day = int(str(day)[::-1])
    
    reversed_year = int(str(year)[::-1]) # a

    o_base36 = base16_to_base36(str(timestamp))
    
    
    calc_base24 = custom_base24encode((year + reversed_year) * (day + r))
    
    ofinal = o_base36 + calc_base24
    
    stringLen =  len(ofinal) # s string len
    
    if (stringLen < 14):
        c = 0
        for c in range(14 - stringLen):
            o += "0"
    else:
        ofinal = ofinal[:14]
    return f"#{ofinal}$"

def decode(data):
    e = format_date_py(data['lastModified'])
    n = e.encode('utf-8')
    r = e.upper().encode('utf-8')
    backend = default_backend()
    cipher = Cipher(algorithms.AES(n), modes.CBC(r), backend=backend)
    decryptor = cipher.decryptor()
    padder = padding.PKCS7(128).unpadder()
    
    decrypted = decryptor.update(base64.b64decode(data['response'])) + decryptor.finalize()
    unpadded = padder.update(decrypted) + padder.finalize()
    
    return json.loads(unpadded.decode('utf-8'))



### Testing the decode

# random user-agent
ua = UserAgent()
headers = {
    'User-Agent': ua.random,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'DNT': '1',  # Do Not Track Request Header
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

session = requests.Session()

# Mimic human behavior by adding delays
time.sleep(random.uniform(1, 5))

url = 'https://itp-ao-sls.infosys-platforms.com/prod/api/court-vision/year/2024/eventId/580/matchId/MS701/pointId/0_0_0'


response = session.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()

    data_json = json.dumps(data)
    decrypted_data = decode(data)
    print("Decrypted data:", decrypted_data)

else:
    print(f"Failed to retrieve data, status code: {response.status_code}")

UTC datetime: 2024-06-10T23:14:32.003000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase36:  400eegtszn
calcbase24:  4mle
Decrypted data: {'courtVisionData': [{'a75': True, 'a76': "Men's Singles", 'a77': 'Rod Laver Arena', 'a78': 1, 'a50': {'1_1_5_1': {'a11': 'false', 'a89': 3.4718000000000004, 'a12': [{'a70': 11.314, 'a71': 0.831, 'a72': 2.852, 'a73': 'hit'}, {'a70': 11.314, 'a71': 0.831, 'a72': 2.852, 'a73': 'peak'}, {'a70': 0.0, 'a71': 0.099, 'a72': 1.057, 'a73': 'net'}, {'a70': -5.17, 'a71': -0.278, 'a72': 0.034, 'a73': 'bounce'}, {'a70': -15.209, 'a71': -1.015, 'a72': 1.203, 'a73': 'last'}], 'a90': 'NA', 'a91': 'Cross Court', 'a92': 'NA', 'a81': '1_1_5_1', 'a13': 'ATPS0AG', 'a14': 'ATPS0AG', 'a15': 'ATPMM58', 'a16': '207 KPH', 'a17': 'NA', 'a18': 'NA', 'a93': 1, 'a94': 1, 'a19': 'NA', 'a20': '3.47 Feet', 'a21': '207 KPH', 'a22': '1.06 Metre', 'a23': 'NA', 'a24': 'NA', 'a95': 'Ace', 'a25': 'NA', 'a96': 'Flat', 'a97': 'DeuceCourt', 'a98': '1', '

## Scrape Matches as JSON


In [11]:
def save_tracking_data(year, league, tournament, num_matches=20):
    '''
    Args:
    -----
    year: [int] - Year of the tournament
    league: [str] - 'atp' or 'wta'
    tournament: [str] - 'rg' (Roland Garros) or 'ao' (Australian Open)
    num_matches: [int] - Number of matches to fetch (default 20)
    '''

    if tournament == 'rg':
        event_id = 520
        prefix = 'SM' if league == 'atp' else 'SD'
    elif tournament == 'ao':
        event_id = 580
        prefix = 'MS' if league == 'atp' else 'WS'
    else:
        raise ValueError("Invalid tournament. Use 'rg' or 'ao'.")

    for i in range(1, num_matches):
        match_num = f'{i:02}'
        match_id = f'{prefix}0{match_num}' if i < 100 else f'{prefix}{match_num}'

        print(match_id)

        api_url = f'https://itp-{tournament}-sls.infosys-platforms.com/prod/api/court-vision/year/{year}/eventId/{event_id}/matchId/{match_id}/pointId/0_0_0'
        get_json_file = requests.get(api_url)

        try:
            tracking_data_json = get_json_file.json()
            tracking_data_json = decode(tracking_data_json)
            tracking_data_json = apply_mapping(tracking_data_json, mapping)

        except ValueError:
            print(f'The Match {match_id} does not exist!')
            continue

        if not tracking_data_json.get('courtVisionData'):
            print(f'The Match {match_id} is empty!')
            continue

        file_name = f'{league}_{tournament.upper()}_year_{year}_{match_id}_tracking_data.json'

        with open(file_name, 'w') as json_file:
            json.dump(tracking_data_json, json_file)


SM001
UTC datetime: 2024-06-10T23:23:18.913000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase36:  400eek1kb7
calcbase24:  4mle
SM002
UTC datetime: 2024-06-10T23:23:22.294000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase36:  400eekau1g
calcbase24:  4mle
SM003
UTC datetime: 2024-06-10T23:23:25.103000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase36:  400eekb37n
calcbase24:  4mle
SM004
The Match SM004 does not exist!
SM005
UTC datetime: 2024-06-10T23:23:29.343000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase36:  400eekbgar
calcbase24:  4mle
SM006
UTC datetime: 2024-06-10T23:23:32.071000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase36:  400eekc86p
calcbase24:  4mle
SM007
UTC datetime: 2024-06-10T23:23:34.128000+00:00
day n:  10
r ?:  1
year i:  2024
reverse year1:  4202
reverse year:  4202
obase

### !Matches ids order can vary in quantity, range and order, so we brute force using a high number of matches

In [ ]:
# Roland Garros
save_tracking_data(year=2024, league="atp", tournament="rg", num_matches=702)
# save_tracking_data(year=2023, league="atp", tournament="rg", num_matches=702)
# save_tracking_data(year=2022, league="atp", tournament="rg", num_matches=702)
# save_tracking_data(year=2021, league="atp", tournament="rg", num_matches=702)
# save_tracking_data(year=2020, league="atp", tournament="rg", num_matches=702)
# save_tracking_data(year=2019, league="atp", tournament="rg", num_matches=702)

save_tracking_data(year=2024, league="wta", tournament="rg", num_matches=702)
# save_tracking_data(year=2023, league="wta", tournament="rg", num_matches=702)
# save_tracking_data(year=2022, league="wta", tournament="rg", num_matches=702)
# save_tracking_data(year=2021, league="wta", tournament="rg", num_matches=702)
# save_tracking_data(year=2020, league="wta", tournament="rg", num_matches=702)
# save_tracking_data(year=2019, league="wta", tournament="rg", num_matches=702)

# Australian Open
save_tracking_data(year=2024, league="atp", tournament="ao", num_matches=702)
# save_tracking_data(year=2023, league="atp", tournament="ao", num_matches=702)
# save_tracking_data(year=2022, league="atp", tournament="ao", num_matches=702)
# save_tracking_data(year=2021, league="atp", tournament="ao", num_matches=702)
# save_tracking_data(year=2020, league="atp", tournament="ao", num_matches=702)
# save_tracking_data(year=2019, league="atp", tournament="ao", num_matches=702)

save_tracking_data(year=2024, league="wta", tournament="ao", num_matches=702)
# save_tracking_data(year=2023, league="wta", tournament="ao", num_matches=702)
# save_tracking_data(year=2022, league="wta", tournament="ao", num_matches=702)
# save_tracking_data(year=2021, league="wta", tournament="ao", num_matches=702)
# save_tracking_data(year=2020, league="wta", tournament="ao", num_matches=702)
# save_tracking_data(year=2019, league="wta", tournament="ao", num_matches=702)
